# 00 — Google Colab Setup

This notebook prepares the environment for the FIQA demographic analysis project.

## Before running

Store the following files in one Google Drive project folder:

- `DiveFace_subset/`
- `DiveFace_subset_annotations.pkl`
- the CR-FIQA model checkpoint, for example `181952backbone.pth`

Change `PROJECT_PATH` and `MODEL_FILENAME` below if your folder or checkpoint uses a different name.

The CR-FIQA source repository is cloned temporarily into the Colab runtime and is not stored inside this project repository.


In [ ]:
# Mount Google Drive and import setup utilities

from pathlib import Path
import subprocess
import sys

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "This notebook is intended to run in Google Colab."
    ) from exc

drive.mount("/content/drive", force_remount=False)


In [ ]:
# Project configuration
# Change these two values when your Google Drive layout is different.

PROJECT_PATH = Path("/content/drive/MyDrive/FIQA_Project")
MODEL_FILENAME = "181952backbone.pth"

IMAGE_FOLDER = PROJECT_PATH / "DiveFace_subset"
ANNOTATION_FILE = PROJECT_PATH / "DiveFace_subset_annotations.pkl"
MODEL_PATH = PROJECT_PATH / MODEL_FILENAME

# The external CR-FIQA repository is cloned into the temporary Colab runtime.
CR_FIQA_PATH = Path("/content/CR-FIQA")
CR_FIQA_REPOSITORY_URL = "https://github.com/fdbtrs/CR-FIQA.git"


In [ ]:
# Validate required project inputs

required_paths = {
    "Project directory": PROJECT_PATH,
    "DiveFace image folder": IMAGE_FOLDER,
    "Annotation file": ANNOTATION_FILE,
    "CR-FIQA checkpoint": MODEL_PATH,
}

missing_paths = [
    f"{name}: {path}"
    for name, path in required_paths.items()
    if not path.exists()
]

if missing_paths:
    missing_text = "\n".join(f"- {item}" for item in missing_paths)
    raise FileNotFoundError(
        "The following required project inputs were not found:\n"
        f"{missing_text}\n\n"
        "Update PROJECT_PATH or MODEL_FILENAME in the configuration cell."
    )

print("All required project inputs were found.")


In [ ]:
# Clone or update CR-FIQA and install dependencies

if not CR_FIQA_PATH.exists():
    print("Cloning the CR-FIQA repository...")
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            CR_FIQA_REPOSITORY_URL,
            str(CR_FIQA_PATH),
        ],
        check=True,
    )
else:
    print("CR-FIQA repository already exists in this runtime.")
    subprocess.run(
        [
            "git",
            "-C",
            str(CR_FIQA_PATH),
            "pull",
            "--ff-only",
        ],
        check=True,
    )

required_packages = [
    "tensorboard",
    "easydict",
    "scikit-learn",
]

print("Installing required packages...")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        *required_packages,
    ],
    check=True,
)

print("Dependencies installed successfully.")


In [ ]:
# Environment summary

import sklearn
import torch

print("===== Environment Check =====")
print(f"Project directory:    {PROJECT_PATH}")
print(f"Image folder:         {IMAGE_FOLDER}")
print(f"Annotation file:      {ANNOTATION_FILE}")
print(f"Model checkpoint:     {MODEL_PATH}")
print(f"CR-FIQA repository:   {CR_FIQA_PATH}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"PyTorch version:      {torch.__version__}")
print(f"CUDA available:       {torch.cuda.is_available()}")

print("\nSetup completed successfully.")
